### 스테이트: 그래프의 상태 표현

In [2]:
import operator
from typing import Annotated
from pydantic import BaseModel, Field

class State(BaseModel):
    query: str = Field(..., description="사용자의 질문")
    current_role: str = Field(default="", description="선정된 답변 역할")
    messages: Annotated[list[str], operator.add] = Field(default=[], description="답변 이력")
    current_judge: bool = Field(default=False, description="품질 검사 결과")
    judgement_reason: str = Field(default="", description="품질 검사 판정 이유")

In [3]:
# 그래프의 스테이트에 대한 타입 정의가 완료되면, StateGraph 클래스에 그 정의를 전달하고 그래프의 인스턴스를 생성
# StateGraph 클래스는 LangGraph에서 그래프 구조 정의를 위해 사용되는 클래스로, 워크플로를 구성하는 노드와 에지를 관리하는 역할을 한다.
from langgraph.graph import StateGraph

workflow = StateGraph(State)

### 노드: 그래프를 구성하는 처리 단위

#### 노드의 지정 방법

In [ ]:
# answering이라는 이름의 노드를 정의
# 노드 내 처리는 answering_node 함수가 수행
workflow.add_node("answering", answering_node)

#### 노드의 구현 방법

In [ ]:
# 노드에 함수를 전달하는 경우, 스테이트 객체를 인자로 받고 변경분을 나타내는 딕셔너리 타입의 객체를 반환하도록 구현
# 예를 들어, 사용자의 질문 내용과 선택된 역할을 기반으로 답변을 생성하는 노드의 구현은 다음과 같다.
from typing import Any

def answering_node(state: State) -> dict[str, Any]:
    query = state.query
    role = state.current_role

    # 사용자 질문 내용과 선택된 역할을 바탕으로 답변을 생성하는 로직
    generated_message = # ...생성 처리...

    # 생성된 답변으로 스테이트 업데이트
    return {"messages": [generated_message]}

In [ ]:
# add_node의 두 번째 인자에 Runnable을 전달하는 경우에도 함수와 마찬가지로 스테이트를 받아 변경분을 반환하는 구현으로 해야 한다.
# 예를 들어 answering_node를 LCEL을 사용해 다시 구현하는 경우는 다음과 같다.
from langchain_core.runnables import RunnablePassthrough

prompt = # ... query와 role을 인자로 받는 ChatPromptTemplate
llm = # ... LangChain의 Chat model

answering_node = (
    RunnablePassthrough.assign(
        query=lambda state: state.query,
        role=lambda state: state.role
    )
    | prompt
    | llm
    | StrOutputParser()
    | RunnablePassthrough.assign(messages=lambda x: [x])
)

In [ ]:
# 여러 필드를 업데이트하는 경우에는 여러 필드 이름과 대응하는 키에 값을 설정한 딕셔너리 타입의 객체를 반환한다.
# 예를 들어 답변의 품질 검사 노드에서 current_judge 필드(품질 검사 결과)와 judgement_reason 필드(품질 검사 판정 이유)를 동시에 업데이트하는 경우에는 다음과 같이 구현
def check_node(state: State) -> dict[str, Any]:
    query = state.query
    message = state.messages[-1]

    #사용자의 질문 내용과 답변 내용에서 품질 검사를 수행하는 처리
    judge = # ...판정 결과...
    reason = # ...이유 생성...

    # 생성된 답변으로 스테이트 업데이트
    return {"current_judge": judge, "judgement_reason": reason}

### 에지: 노드 간의 연결

#### 1. 엔트리 포인트

In [ ]:
# 그래프의 시작 노드를 지정하는 에지
# set_entry_point 함수로 시작 노드가 될 노드 이름을 문자열로 지정
# set_entry_point 함수를 사용하는 방법 외에도 시작 노드를 나타내는 START 노드를 에지로 사용하는 방법도 있지만, 이 책에서는 set_entry_point 함수를 사용하는 형태로 통일
workflow.set_entry_point("selection")
# START 노드를 이용한 경우의 코드 예
workflow.add_edge(START, "selection")

#### 2. 에지

In [ ]:
# 특정 노드에서 다른 노드로 무조건 전이하는 에지
# add_edge 함수로 설정
# 첫 번째 인자에 전이 시작 노드 이름, 두 번째 인자에 전이 대상 노드 이름을 각각 문자열로 지정

# selection 노드에서 answering 노드로 에지를 연결
workflow.add_edge("selection", "answering")

#### 3. 조건부 에지

In [ ]:
# 조건에 기반해 전이할 노드를 결정하는 에지
# add_conditional_edges 함수로 설정
# 첫 번째 인자에 전이 시작 노드 이름을 문자열로 지정하고, 두 번째 인자에 어떤 값을 반환하는 함수를 설정
# 세 번째 인자에는 두 번째 인자에서 반환되는 값에 대응하는 전이 대상 노드 이름과의 매핑을 딕셔너리 타입의 객체로 설정
# state.current_judge의 값이 True이면 종점 노드를 나타내는 END로, False이면 selection 노드로 전이하도록 설정
# END는 LangGraph에서 종점 노드를 나타내는 내장 상수
from langgraph.graph import END
# check 노드에서 에지를 연결
workflow.add_conditional_edges(
    "check",
    lambda state: state.current_judge,
    {True: END, False: "selection"}
)

### 컴파일된 그래프

In [ ]:
# 정의된 그래프는 compile 함수를 통해 실행 가능한 CompiledGraph 인스턴스로 변환된다.
compiled = workflow.compile()

#### invoke 함수

In [ ]:
# invoke 함수를 사용하면 그래프 내의 모든 처리가 실행된 후 최종 값이 반환된다.
initial_state = State(query="생성 AI에 관해 알려주세요")
result = compiled.invoke(initial_state)

#### ainvoke 함수

In [ ]:
# 비동기 함수로 실행할 수 있다.
# LangChain/LangGraph에서는 함수 이름 앞에 a가 붙은 것(ainvoke, astream 등)은 Python의 비동기 메커니즘(asyncio 등)을 사용해 비동기적으로 호출할 수 있다.
# LLM을 사용한 처리는 시간이 오래 걸리는 경우가 많으므로 여러 처리를 병렬로 실행하기 위해 비동기 메커니즘이 활용
initial_state = State(query="생성 AI에 관해 알려주세요")
result = await compiled.ainvoke(initial_state)

#### stream 함수

In [ ]:
# stream 함수를 사용하면 노드 실행 시 스테이트를 순차적으로 가져올 수 있다.
initial_state = State(query="생성 AI에 관해 알려주세요")
for step in compiled.stream(initial_state):
    print(step)